# Latent RAG — Factorial Experiment Runner
Compares traditional vs. latent-space RAG across 4 controlled experiments.

| # | Retriever | Generator | Research question |
|---|---|---|---|
| 1 | BGE (dense) | Qwen (text) | Traditional baseline |
| 2 | BGE (dense) | T5Gemma (latent) | Can latent work with good retrieval? |
| 3 | T5Gemma (latent) | Qwen (text) | Does the T5 encoder suck at retrieval? |
| 4 | T5Gemma (latent) | T5Gemma (latent) | Does latent help at all over T5 text? |

## Clone private repo
Use a GitHub Personal Access Token with read access to this repo.

In [1]:
import getpass
import os
import subprocess

os.chdir('/content')

token = getpass.getpass('GitHub token (repo read access): ')
result = subprocess.run(
    ['git', 'clone', f'https://{token}@github.com/zahiTouqan/latent-rag.git'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('STDOUT:', result.stdout.replace(token, '***'))
    print('STDERR:', result.stderr.replace(token, '***'))
    raise SystemExit('Clone failed')
else:
    print('Clone successful')

Clone successful


In [2]:
%cd /content/latent-rag
!git fetch
!git checkout v3
!git pull origin v3
!python -m pip install -U pip
!pip install -r requirements.txt
!pip install "datasets>=2.14,<3.0"
!pip install -q --upgrade transformers

/content/latent-rag
Branch 'v3' set up to track remote branch 'v3' from 'origin'.
Switched to a new branch 'v3'
From https://github.com/zahiTouqan/latent-rag
 * branch            v3         -> FETCH_HEAD
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 16.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.7 MB/s  0:00:00m0:00:0100:01
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [faiss-cpu]
    Found existing installation: datasets 4.0.0━━━━━━━━━━━━━━━ 1/3 [faiss-cpu]
    Uninstalling datasets-4.0.0:━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [faiss-cpu]

## Download some NQ data
Streams `facebook/dpr` NQs + related documents. Adjust the slice `[:100]` to change dataset size.

In [3]:
import json, gzip, urllib.request

url = "https://dl.fbaipublicfiles.com/dpr/data/retriever/biencoder-nq-dev.json.gz"
with urllib.request.urlopen(url) as response:
    with gzip.open(response, 'rt', encoding='utf-8') as f:
        data = json.load(f)

sample = data[:100]

Example DPR record:

In [4]:
import pprint
first = {k: v for k, v in sample[0].items() if k != 'negative_ctxs'}
first['positive_ctxs'] = [{'title': c['title'], 'passage_id': c['passage_id']} for c in first['positive_ctxs']]
pprint.pprint(first, width=120)

{'answers': ['Linda Davis'],
 'dataset': 'nq_dev_psgs_w100',
 'hard_negative_ctxs': [{'passage_id': '14525568',
                         'score': 14.678405,
                         'text': 'song. According to the lyrics of "Why Don\'t You Love Me", Knowles impersonates a '
                                 'woman who questions her love interest about the reason for which he does not value '
                                 'her fabulousness, convincing him she\'s the best thing for him as she sings: "Why '
                                 "don't you love me... when I make me so damn easy to love?... I got beauty... I got "
                                 'class... I got style and I got ass...". The singer further tells her love interest '
                                 'that the decision not to choose her is "entirely foolish". Originally released as a '
                                 'pre-order bonus track on the deluxe edition of "I Am...',
                         'title': "Why

## Build mini corpus from NQ passages
Collects all `positive_ctxs` + `hard_negative_ctxs` from the sampled questions.

In [5]:
import json
from pathlib import Path

CORPUS_OUT = Path("data/passages.jsonl")
CORPUS_OUT.parent.mkdir(parents=True, exist_ok=True)

corpus = {}
for item in sample:
    for p in item["positive_ctxs"] + item["hard_negative_ctxs"]:
        pid = p["passage_id"]
        if pid not in corpus:
            corpus[pid] = {
                "id": pid,
                "text": f"{p['title']}: {p['text']}",
                "doc_id": p["title"]
            }

with CORPUS_OUT.open("w", encoding="utf-8") as f:
    for record in corpus.values():
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(corpus)} passages to {CORPUS_OUT}")

Saved 10010 passages to data/passages.jsonl


## GPU check & HuggingFace login

In [6]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")

CUDA available: True


In [7]:
from huggingface_hub import login

token = getpass.getpass('HF_TOKEN')
login(token)

## Build BGE index (for experiments 1 & 2)
Embedding model: `BAAI/bge-base-en-v1.5` — dense retrieval with [CLS]-token pooling.

In [8]:
BGE_INDEX_DIR = "/content/index_bge"
!python build_index.py --corpus_path {CORPUS_OUT} --index_dir {BGE_INDEX_DIR} --retriever_type bge

Loaded 10010 passages from data/passages.jsonl
Loading BGE embedding model: BAAI/bge-base-en-v1.5
config.json: 100% 777/777 [00:00<00:00, 3.12MB/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 1.98MB/s]
vocab.txt: 232kB [00:00, 70.1MB/s]
tokenizer.json: 711kB [00:00, 92.4MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 692kB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 438M/438M [00:02<00:00, 167MB/s] 
Loading weights: 100% 199/199 [00:00<00:00, 1347.62it/s]
Embedding 10010 passages (40 batches)...
  batch 25/40
  batch 40/40
Saved bge index to /content/index_bge


In [9]:
!ls -lh {BGE_INDEX_DIR}

total 37M
-rw-r--r-- 1 root root  224 May  4 15:33 config.json
-rw-r--r-- 1 root root  30M May  4 15:33 index.faiss
-rw-r--r-- 1 root root 6.8M May  4 15:33 metadata.jsonl


## Build latent (T5) index (for experiments 3 & 4)
Embedding model: `google/t5gemma-2-270m-270m` — mean-pooled encoder latents + safetensors storage.
Stores both FAISS vectors and full sequence latent matrices.
Warning: safetensors file is large (~1.8 GB for 10k passages).

In [10]:
LATENT_INDEX_DIR = "/content/index_latent"
!python build_index.py --corpus_path {CORPUS_OUT} --index_dir {LATENT_INDEX_DIR} --retriever_type latent

Loaded 10010 passages from data/passages.jsonl
Loading latent embedding model (encoder only): google/t5gemma-2-270m-270m
config.json: 6.03kB [00:00, 4.72MB/s]
tokenizer_config.json: 100% 830/830 [00:00<00:00, 4.31MB/s]
tokenizer.json: 100% 33.4M/33.4M [00:01<00:00, 20.6MB/s]
special_tokens_map.json: 100% 662/662 [00:00<00:00, 3.46MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 1.57G/1.57G [00:16<00:00, 96.9MB/s] 
Loading weights: 100% 911/911 [00:00<00:00, 5129.20it/s]
generation_config.json: 100% 195/195 [00:00<00:00, 913kB/s]
Embedding 10010 passages (157 batches)...
  batch 10/157
  batch 20/157
  batch 30/157
  batch 40/157
  batch 50/157
  batch 60/157
  batch 70/157
  batch 80/157
  batch 90/157
  batch 100/157
  batch 110/157
  batch 120/157
  batch 130/157
  batch 140/157
  batch 150/157
  batch 157/157
Saving latent sequences to safetensors...
Saved latent index to /content/index_latent


In [11]:
!ls -lh {LATENT_INDEX_DIR}

total 1.8G
-rw-r--r-- 1 root root  232 May  4 15:38 config.json
-rw-r--r-- 1 root root  25M May  4 15:37 index.faiss
-rw-r--r-- 1 root root 1.8G May  4 15:38 latents.safetensors
-rw-r--r-- 1 root root 6.8M May  4 15:37 metadata.jsonl


## Build eval file
Construct `data/eval.jsonl` from the DPR dev set. `relevant_ids` are document titles for document-level recall.

In [12]:
import json
from pathlib import Path

EVAL_OUT = Path("data/eval.jsonl")

with EVAL_OUT.open("w", encoding="utf-8") as f:
    for item in sample:
        record = {
            "question": item["question"],
            "answer": item["answers"],
            "relevant_ids": [p["title"] for p in item["positive_ctxs"]]
        }
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(sample)} eval examples to {EVAL_OUT}")

Saved 100 eval examples to data/eval.jsonl


## Run factorial experiments
Each cell runs one experiment. Results are saved to `results/` and can be aggregated after all 4 finish.

### Experiment 1: BGE + Text (traditional baseline)
BGE dense retrieval → prompted text generation with Qwen.

In [13]:
!python3 evaluate.py \
    --index_dir {BGE_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode bge+text \
    --top_k 5

Loaded 100 samples from data/eval.jsonl
Loading BGE embedding model: BAAI/bge-base-en-v1.5
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 199/199 [00:01<00:00, 111.54it/s]
Loading text generator model: Qwen/Qwen3.5-0.8B
config.json: 2.91kB [00:00, 711kB/s]
tokenizer_config.json: 16.7kB [00:00, 14.2MB/s]
vocab.json: 6.72MB [00:00, 38.4MB/s]
merges.txt: 3.35MB [00:00, 12.4MB/s]
tokenizer.json: 100% 12.8M/12.8M [00:01<00:00, 9.91MB/s]
chat_template.jinja: 7.75kB [00:00, 20.2MB/s]
model.safetensors.index.json: 50.9kB [00:00, 112MB/s]
model.safetensors-00001-of-00001.safeten(…): 100% 1.75G/1.75G [00:20<00:00, 84.8MB/s]
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100% 320/320 [00:00<00:00, 4874.28it/s]
Pipeline mode: 

### Experiment 2: BGE + Latent
BGE dense retrieval → T5Gemma re-encodes retrieved passages on the fly → decoder bypass.

In [14]:
!python3 evaluate.py \
    --index_dir {BGE_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode bge+latent \
    --top_k 5

Loaded 100 samples from data/eval.jsonl
Loading BGE embedding model: BAAI/bge-base-en-v1.5
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 199/199 [00:00<00:00, 1799.17it/s]
Loading latent generator model (Seq2Seq): google/t5gemma-2-270m-270m
Loading weights: 100% 911/911 [00:01<00:00, 797.20it/s]
Pipeline mode: bge+latent | Index: 10010 passages from data/passages.jsonl
100% 100/100 [09:53<00:00,  5.94s/it]

=== Results ===
  em                                  0.0000
  f1                                  0.0124
  generated_tokens                    118.4700
  generation_time_s                   5.5894
  latency_p50_ms                      6320.0147
  latency_p95_ms                      6826.4862
  recall@5                            0.6356
  retrieval_time_s                    0.0338
  total_time_s                        5.9346

Saved to results/results_eval_bge+latent_20260504_155947.json


### Experiment 3: T5 + Text
T5 encoder mean-pooled retrieval (likely poor) → Qwen text generation.
Isolates retrieval quality of T5 encoder.

In [15]:
!python3 evaluate.py \
    --index_dir {LATENT_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode t5+text \
    --top_k 5

Loaded 100 samples from data/eval.jsonl
Loading latent embedding model (encoder only): google/t5gemma-2-270m-270m
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 911/911 [00:00<00:00, 17289.72it/s]
Loading text generator model: Qwen/Qwen3.5-0.8B
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100% 320/320 [00:00<00:00, 5180.45it/s]
Pipeline mode: t5+text | Index: 10010 passages from data/passages.jsonl
100% 100/100 [10:04<00:00,  6.05s/it]

=== Results ===
  em                                  0.0000
  f1                                  0.0120
  generated_tokens                    113.2600
  generation_time_s                   5.9667
  latency_p50_ms                      6510.1256
  latency_p95_ms                   

### Experiment 4: T5 + Latent (full latent pipeline)
T5 encoder retrieval + pre-stored safetensors latents → decoder bypass. The current state of the repo.

In [16]:
!python3 evaluate.py \
    --index_dir {LATENT_INDEX_DIR} \
    --eval_path {EVAL_OUT} \
    --retrieval_id_field source_doc_id \
    --mode t5+latent \
    --top_k 5

Loaded 100 samples from data/eval.jsonl
Loading latent embedding model (encoder only): google/t5gemma-2-270m-270m
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 911/911 [00:00<00:00, 20046.96it/s]
Loading latent generator model (Seq2Seq): google/t5gemma-2-270m-270m
Loading weights: 100% 911/911 [00:00<00:00, 5104.18it/s]
Pipeline mode: t5+latent | Index: 10010 passages from data/passages.jsonl
100% 100/100 [08:46<00:00,  5.26s/it]

=== Results ===
  em                                  0.0000
  f1                                  0.0029
  generated_tokens                    113.7700
  generation_time_s                   5.1494
  latency_p50_ms                      5629.2197
  latency_p95_ms                      6340.2500
  recall@5                            0.0773
  retrieval_time_s                    0.0653
  total_time_s                        5.2611

Saved to results/results_eval_t5+latent_20260504_161920.json


## Compare results
Collects all result files from `results/` and displays a side-by-side comparison table.

In [17]:
import json
from pathlib import Path

results_dir = Path("results")
files = sorted(results_dir.glob("results_*.json"), key=lambda p: p.stat().st_mtime)

COLUMNS = ["em", "f1", "recall@5", "latency_p50_ms", "latency_p95_ms"]

rows = []
for result_file in files:
    with result_file.open() as f:
        data = json.load(f)
    mode = data["config"]["mode"]
    metrics = data["metrics"]
    rows.append((mode, metrics))

if not rows:
    print("No result files found in results/")
else:
    header = f"{'Experiment':<14s}" + "".join(f"{c:>15s}" for c in COLUMNS)
    print(header)
    print("-" * len(header))
    for mode, m in rows:
        vals = "".join(f"{m.get(c, 0):>15.4f}" for c in COLUMNS)
        print(f"{mode:<14s}{vals}")
    if len(rows) < 4:
        print(f"\n({len(rows)}/4 experiments completed)")

Experiment                 em             f1       recall@5 latency_p50_ms latency_p95_ms
-----------------------------------------------------------------------------------------
bge+text               0.0400         0.0717         0.6356      6659.5704      7225.2928
bge+latent             0.0000         0.0124         0.6356      6320.0147      6826.4862
t5+text                0.0000         0.0120         0.0773      6510.1256      7073.4844
t5+latent              0.0000         0.0029         0.0773      5629.2197      6340.2500


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
!ls -lh /content/latent-rag/results
!cp -r /content/latent-rag/results /content/drive/MyDrive/

total 552K
-rw-r--r-- 1 root root 132K May  4 15:59 results_eval_bge+latent_20260504_155947.json
-rw-r--r-- 1 root root 140K May  4 15:49 results_eval_bge+text_20260504_154929.json
-rw-r--r-- 1 root root 132K May  4 16:19 results_eval_t5+latent_20260504_161920.json
-rw-r--r-- 1 root root 145K May  4 16:10 results_eval_t5+text_20260504_161012.json


In [23]:
from transformers import AutoConfig
config = AutoConfig.from_pretrained("google/t5gemma-2-270m-270m")
print(f"decoder_start_token_id from config: {config.decoder_start_token_id}")
print(f"bos_token_id: {config.bos_token_id}")
print(f"pad_token_id: {config.pad_token_id}")
print(f"eos_token_id: {config.eos_token_id}")
# Also check generation_config
gen_config = config
if hasattr(config, 'generation_config'):
    print(f"generation_config.decoder_start_token_id: {config.generation_config.decoder_start_token_id}")

AttributeError: 'T5Gemma2Config' object has no attribute 'decoder_start_token_id'